# Tutorial — Capital econômico de carteira (`yggdrasil.credit_risk.capital`)

**Capital econômico** é o capital que a instituição estima ser necessário para absorver as **perdas
inesperadas** da carteira de crédito, em um horizonte de 1 ano e num nível de confiança compatível com o
apetite de risco (ex.: 99,9%). Ele responde a uma pergunta diferente da provisão:

- a **perda esperada** (EL) é um custo previsível, coberto por **provisão** (ECL, sob IFRS 9 / Resolução
  CMN 4.966) e precificado no *spread*;
- a **perda inesperada** (UL) é a variabilidade em torno dessa média — só pode ser absorvida por **capital**.

A relação central, no nível de confiança $q$ e horizonte de 1 ano, é:

$$CE = \mathrm{VaR}_q(L) - EL, \qquad EL = \sum_i PD_i \cdot LGD_i \cdot EAD_i$$

Este tutorial percorre **o método** de ponta a ponta, na ordem em que um modelo de capital se constrói
(guia de construção): contrato de dados → **v1** analítico (ASRF/Vasicek) → insumos (correlações,
parâmetros *downturn*) → **v2** simulação de Monte Carlo multifatorial → alocação e RAROC → validação e
registro. A recomendação prática do guia é justamente essa: **uma v1 simples e completa antes de sofisticar**.

> Contexto regulatório: Resolução CMN 4.557/2017 (ICAAP / Pilar 2) e 4.966/2021 (ponto de partida dos parâmetros).

## 0. Setup

O núcleo depende só de `numpy`/`pandas`/`scipy`/`matplotlib` (tudo no core do pacote). As figuras usam `matplotlib`; o registro no MLflow (seção 9) usa o *file store* local.

In [ ]:
# --- Bootstrap: torna o pacote `yggdrasil` importável a partir do repositório,
# sem `pip install`. Procura a raiz do repo (a pasta que contém `yggdrasil/`) por
# vários âncoras — o caminho do próprio notebook (VS Code expõe `__vsc_ipynb_file__`),
# o diretório atual, os diretórios do sys.path e, no Databricks, o caminho via
# dbutils — subindo até achá-la, e a insere no sys.path. Cobre Jupyter/VS Code
# local e Databricks; se o pacote já estiver importável, é inócuo.
import sys
from pathlib import Path

def _find_yggdrasil_root():
    _anchors = []
    for _n in ("__vsc_ipynb_file__", "__file__", "__session__"):
        _v = globals().get(_n)
        if _v:
            _anchors.append(Path(str(_v)))
    _anchors.append(Path.cwd())
    _anchors += [Path(_p) for _p in sys.path if _p not in ("", ".")]
    for _a in _anchors:
        try:
            _a = _a.resolve()
        except Exception:
            continue
        for _b in (_a, *_a.parents):
            if (_b / "yggdrasil" / "__init__.py").is_file():
                return _b
    try:  # fallback Databricks: caminho do próprio notebook
        _nbp = (dbutils.notebook.entry_point.getDbutils()  # noqa: F821
                .notebook().getContext().notebookPath().get())
        for _pref in ("/Workspace", ""):
            for _b in Path(_pref + _nbp).parents:
                if (_b / "yggdrasil" / "__init__.py").is_file():
                    return _b
    except Exception:
        pass
    return None

_ygg_root = _find_yggdrasil_root()
if _ygg_root and str(_ygg_root) not in sys.path:
    sys.path.insert(0, str(_ygg_root))


In [ ]:
%matplotlib inline
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")   # MLflow 3.x local (seção 9)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 30)
rng = np.random.default_rng(42)

from yggdrasil.credit_risk import capital
from yggdrasil.credit_risk.capital import Portfolio, Segment
print("capital econômico — API com", len([x for x in dir(capital) if not x.startswith('_')]), "símbolos")

## 1. O contrato de dados: `Segment` e `Portfolio`

A **segmentação é a espinha dorsal** do modelo: cada segmento deve ser homogêneo em comportamento de risco.
Um `Segment` carrega os parâmetros de **capital econômico** — que **não** são os *point-in-time* da provisão:

| Parâmetro | Provisão (ECL / IFRS 9) | **Capital econômico** |
|---|---|---|
| `pd` | PIT, *forward-looking* | **TTC** (média de longo prazo do ciclo), 1 ano |
| `lgd` | melhor estimativa PIT | **downturn** (calibrada a estresse) |
| `ead` | melhor estimativa dos saques | **CCF/LEQ downturn** (saque pré-*default*) |
| `rho` | não é insumo | **insumo central**: correlação de ativos intra-segmento |

Cada segmento aponta para um **fator sistêmico** (`factor`); a `Portfolio` junta os segmentos e a
**matriz de correlação entre fatores**, que é o que produz o **benefício de diversificação** entre produtos.

Os `presets` de produto (`apply_preset`) já trazem valores plausíveis de `rho`, `lgd_vol` e a classe de
Basileia para cartão, consignado e veículos (Tabela 3 do guia).

In [ ]:
from yggdrasil.credit_risk.capital import preset, apply_preset, presets_frame

# Particularidades por produto embutidas no pacote (Tabela 3 do guia):
presets_frame()[["produto", "basel_class", "parametro_critico", "nivel_pd", "nivel_lgd", "correlacao_ciclo"]]

In [ ]:
# Carteira de varejo com 3 produtos (5 segmentos homogêneos).
# Note: pd = TTC, lgd = downturn, ead já com CCF downturn; rho estimado internamente (seção 3).
segmentos = [
    # cartão: PD alta e ciclo-sensível, LGD alta (sem garantia), EAD via CCF (rotativo)
    Segment("cartao_revolver_alto", pd=0.11, lgd=0.78, ead=6_000_000, rho=0.11,
            n_obligors=30_000, product="cartao",     factor="cartao"),
    Segment("cartao_transactor",    pd=0.03, lgd=0.72, ead=4_000_000, rho=0.09,
            n_obligors=50_000, product="cartao",     factor="cartao"),
    # consignado: PD baixa/estável (desconto em folha), LGD baixa (margem/seguro); pouco ciclo-sensível
    Segment("consig_inss",          pd=0.008, lgd=0.28, ead=10_000_000, rho=0.04,
            n_obligors=80_000, product="consignado", factor="consignado"),
    Segment("consig_privado",       pd=0.020, lgd=0.35, ead=5_000_000, rho=0.05,
            n_obligors=40_000, product="consignado", factor="consignado"),
    # veículos: LGD é o centro do modelo — garantia real, LGD estocástica (preço de usados no ciclo)
    Segment("veiculos_ltv_alto",    pd=0.035, lgd=0.45, ead=8_000_000, rho=0.08,
            n_obligors=35_000, product="veiculos",   factor="veiculos", lgd_vol=0.22),
]

# Matriz de correlação ENTRE fatores (cartão/consignado/veículos). É ela que dá diversificação.
factor_corr = np.array([
    [1.00, 0.25, 0.35],   # cartao
    [0.25, 1.00, 0.20],   # consignado
    [0.35, 0.20, 1.00],   # veiculos
])
carteira = Portfolio(segmentos, factor_corr=factor_corr,
                     factor_names=["cartao", "consignado", "veiculos"], name="varejo_pf")

print(f"EAD total = {carteira.total_ead():,.0f}   |   EL da carteira = {carteira.expected_loss():,.0f}")
carteira.summary()

## 2. Versão 1 — ASRF / Vasicek analítico

O modelo **assintótico de fator único** (ASRF) é o modelo de Merton levado ao limite de uma carteira
granular: sob um único fator sistêmico, a perda condicional ao quantil $q$ tem **forma fechada**. O capital
por unidade de exposição é

$$K = LGD \cdot \Big[\, N\!\Big(\tfrac{N^{-1}(PD) + \sqrt{\rho}\, N^{-1}(q)}{\sqrt{1-\rho}}\Big) - PD \,\Big]$$

— exatamente a fórmula do IRB de Basileia, mas aqui com $\rho$ **estimado internamente**. É **transparente,
de custo nulo e aditivo** (o capital da carteira é a soma dos capitais dos segmentos). A contrapartida: um
único fator **não** captura diversificação entre produtos. É o ponto de partida e o *benchmark*.

In [ ]:
asrf = carteira.asrf_capital(q=0.999)
print(asrf.summary().to_string(index=False))
print("\nO capital agregado é a SOMA dos capitais por segmento (aditividade do ASRF):")
asrf.per_segment[["segmento", "produto", "PD", "LGD", "rho", "cond_pd", "capital"]]

## 3. Insumos: estimar a correlação de ativos $\rho$

$\rho$ é o parâmetro que **mais move o resultado** e o que **menos dados** temos para estimar. Estima-se da
**série histórica de taxas de *default*** de cada segmento (idealmente cobrindo um ciclo completo). O pacote
traz três abordagens do guia: **método dos momentos**, **máxima verossimilhança** (mistura de Vasicek) e
**modelo macro-fatorial**. Boa prática: estimar por mais de um método e comparar com os $\rho$ regulatórios
do IRB como teste de sanidade.

Abaixo geramos uma série sintética de *defaults* com $\rho$ conhecido e recuperamos o valor.

In [ ]:
from yggdrasil.credit_risk.capital import (
    asset_correlation_moments, asset_correlation_mle, basel_correlation,
)
from scipy.stats import norm

# Série mensal sintética de defaults de um segmento sob Vasicek, com rho "verdadeiro" = 0.08.
T, n_obl, pd_true, rho_true = 240, 800, 0.03, 0.08
Y = rng.standard_normal(T)                                            # fator sistêmico (baixo = ruim)
p_cond = norm.cdf((norm.ppf(pd_true) - np.sqrt(rho_true) * Y) / np.sqrt(1 - rho_true))
defaults = rng.binomial(n_obl, p_cond)                               # nº de defaults por período
taxa_default = defaults / n_obl

print(f"rho verdadeiro            = {rho_true:.3f}")
print(f"momentos (série de taxas) = {asset_correlation_moments(taxa_default):.3f}")
print(f"MLE (mistura de Vasicek)  = {asset_correlation_mle(defaults, np.full(T, n_obl)):.3f}")
print(f"IRB Basileia (other_retail, pd={pd_true}) = {basel_correlation(pd_true, 'other_retail'):.3f}  [sanidade]")

E a **matriz de correlação entre fatores** (produtos) sai das séries de *default* conjuntas — é ela que
a validação estressa, pois correlações sobem em crise.

In [ ]:
from yggdrasil.credit_risk.capital import factor_correlation_matrix, nearest_correlation, is_positive_definite

# Três séries de default (uma por produto) com fatores correlacionados entre si.
Z = rng.standard_normal((T, 3))
L = np.linalg.cholesky(np.array([[1, .3, .4], [.3, 1, .2], [.4, .2, 1]]))
Yp = Z @ L.T
series = {}
for j, (nome, pdp, rp) in enumerate([("cartao", .06, .10), ("consignado", .012, .04), ("veiculos", .03, .08)]):
    pj = norm.cdf((norm.ppf(pdp) - np.sqrt(rp) * Yp[:, j]) / np.sqrt(1 - rp))
    series[nome] = rng.binomial(1000, pj) / 1000
M = factor_correlation_matrix(pd.DataFrame(series))
print("matriz entre fatores é positiva-definida?", is_positive_definite(M))
pd.DataFrame(M, index=series.keys(), columns=series.keys())

## 4. Versão 2 — Monte Carlo multifatorial

É o **padrão de mercado** para capital econômico interno. Em cada cenário: sorteiam-se os **fatores
sistêmicos correlacionados**, condicionam-se as PDs de cada segmento ao cenário (fórmula de Vasicek),
agregam-se as perdas. Repetindo centenas de milhares de vezes, obtém-se a **distribuição empírica completa**
de perdas — da qual saem VaR, ES e as contribuições de cada produto.

Uma sanidade importante (guia): com **um único fator** e carteira **granular**, a simulação **reproduz o
ASRF**. Com **múltiplos fatores**, o capital fica **menor** que a soma dos capitais isolados — a diferença é
o **benefício de diversificação**.

In [ ]:
sim = carteira.simulate(n_scenarios=200_000, q=0.999, seed=7)   # v2
print(sim.summary().to_string(index=False))
print(f"\nCE (v2, Monte Carlo) = {sim.economic_capital():,.0f}"
      f"   vs   CE (v1, ASRF) = {asrf.economic_capital:,.0f}")

### A distribuição de perdas (Figura 1 do guia)

A distribuição é fortemente **assimétrica à direita**: a provisão cobre a EL; o capital econômico cobre a
perda inesperada de EL até o VaR; a cauda além do VaR é o risco aceito pelo apetite de risco.

In [ ]:
fig = capital.report.plot_loss_distribution(sim, q=0.999)
fig

### LGD estocástica e correlação adversa PD–LGD (veículos)

Em produtos com garantia, a severidade é **correlacionada ao ciclo**: em recessão os *defaults* sobem e o
preço dos usados cai ao mesmo tempo. Ligando `stochastic_lgd=True` com `pd_lgd_corr > 0`, o capital **sobe** —
o modelo passa a capturar esse risco que a LGD determinística ignora.

In [ ]:
sim_det   = carteira.simulate(200_000, q=0.999, seed=7, stochastic_lgd=False)
sim_stoch = carteira.simulate(200_000, q=0.999, seed=7, stochastic_lgd=True, pd_lgd_corr=0.5)
print(f"CE com LGD determinística        = {sim_det.economic_capital():,.0f}")
print(f"CE com LGD estocástica (adversa) = {sim_stoch.economic_capital():,.0f}")

## 5. Alocação de capital (Euler) e RAROC

Calculado o capital total, devolve-se aos produtos de forma **coerente**: a soma das parcelas iguala o total,
e cada parcela reflete a contribuição do segmento ao **risco conjunto** (não ao isolado). O método padrão é a
**alocação de Euler** — em simulação, a perda média do segmento **condicionada aos cenários da cauda**.
Segmentos que perdem justamente nos cenários ruins da carteira recebem mais capital; os descorrelacionados
recebem menos. Com o capital alocado, fecha-se o **RAROC** (resultado ajustado ao risco ÷ capital).

In [ ]:
alloc = sim.allocate(metric="es")   # contribuição à cauda (ES), estável para alocação
alloc[["segmento", "EL", "capital_alocado", "capital_isolado", "beneficio_diversificacao", "share_capital"]]

In [ ]:
div = sim.diversification_benefit()
print(f"capital isolado (soma dos CE por segmento) = {div['capital_isolado']:,.0f}")
print(f"capital integrado (carteira)               = {div['capital_integrado']:,.0f}")
print(f"benefício de diversificação                = {div['beneficio_diversificacao']:,.0f}"
      f"  ({div['beneficio_pct']:.1%})")
fig = capital.report.plot_allocation(alloc); fig

In [ ]:
# RAROC por produto: receita e custo hipotéticos por segmento (base única de comparação).
from yggdrasil.credit_risk.capital import raroc_table
receitas = {"cartao_revolver_alto": 900_000, "cartao_transactor": 260_000,
            "consig_inss": 520_000, "consig_privado": 300_000, "veiculos_ltv_alto": 430_000}
custos   = {k: v * 0.30 for k, v in receitas.items()}
raroc_table(alloc, receitas, custos, hurdle_rate=0.15)[
    ["segmento", "capital_alocado", "RAROC", "cria_valor", "precificacao_minima"]]

## 6. Validação: benchmark, sensibilidades, estresse e Pilar 1

O *backtesting* direto do quantil 99,9% é **impossível** por construção; valida-se o **corpo** da distribuição,
a **qualidade dos parâmetros** e a **razoabilidade das correlações** por *benchmark* e estresse.

In [ ]:
from yggdrasil.credit_risk.capital import (
    benchmark, sensitivity, correlation_stress, pillar1_comparison, convergence,
)

# Benchmark independente: ASRF (aditivo) vs Monte Carlo (diversifica) vs CreditRisk+ (atuarial).
benchmark(carteira, q=0.999, n_scenarios=200_000, seed=7)

In [ ]:
# Sensibilidades univariadas (ASRF, barato e determinístico): PD, LGD, rho ±10% e ±20%.
sensitivity(carteira, q=0.999, shocks=(-0.20, -0.10, 0.10, 0.20), params=("pd", "lgd", "rho"))

In [ ]:
# Estresse de correlações entre fatores (+25%, +50%): correlações sobem em crise.
correlation_stress(carteira, deltas=(0.25, 0.50), n_scenarios=150_000, q=0.999, seed=7)

In [ ]:
# Capital econômico x capital regulatório de Pilar 1 (IRB) por classe de exposição.
classes = {"cartao_revolver_alto": "revolving", "cartao_transactor": "revolving",
           "consig_inss": "other_retail", "consig_privado": "other_retail",
           "veiculos_ltv_alto": "other_retail"}
pillar1_comparison(carteira, classes, n_scenarios=150_000, seed=7)

In [ ]:
# Convergência: estabilidade do VaR/ES conforme cresce o nº de cenários (com repetições, o ruído da cauda).
conv = convergence(carteira, n_grid=(5_000, 20_000, 50_000, 150_000), q=0.999, seed=7, n_repeats=3)
display(conv)
fig = capital.report.plot_convergence(conv); fig

## 7. CreditRisk+ e migração (motores alternativos)

**CreditRisk+** é a abordagem **atuarial** (Poisson com intensidade gama, distribuição por recursão de
Panjer) — rápido e sem simulação, ótimo como *benchmark*. **CreditMetrics / migração** captura perdas por
**migração de rating/estágio** (não só *default*), útil para conectar capital e provisão (Stage 1/2/3).

In [ ]:
from yggdrasil.credit_risk.capital import creditrisk_plus, MigrationModel, two_state_matrix

crp = creditrisk_plus(carteira, sigma=0.5)   # sigma = volatilidade sistêmica da taxa de default
print(f"CreditRisk+   EL = {crp.el:,.0f}   VaR99.9 = {crp.var(0.999):,.0f}   CE = {crp.economic_capital():,.0f}")

# Migração default-only (2 estados) reproduz o modo Bernoulli e permite estágios IFRS 9.
tm, ratings, _ = two_state_matrix(pd=0.03)
mm = MigrationModel(tm, ratings, values=np.array([1.0, 1.0 - 0.45]), rho=0.08)
ld = mm.simulate(np.array([1_000_000] * 500), np.zeros(500, dtype=int), n_scenarios=100_000, q=0.999, seed=1)
print(f"Migração      EL = {ld.el:,.0f}   VaR99.9 = {ld.var(0.999):,.0f}")

## 8. Registro no MLflow

Capital econômico é um modelo de **alto impacto** — a governança exige versionamento e reprodutibilidade
(manual do modelo, inventário, recalibração). `log_capital_run` registra parâmetros, métricas (EL/VaR/ES/CE,
diversificação, ASRF), a alocação e as figuras num *run* do MLflow.

In [ ]:
import mlflow, tempfile
from yggdrasil.credit_risk.capital import log_capital_run

with tempfile.TemporaryDirectory() as d:
    mlflow.set_tracking_uri((__import__("pathlib").Path(d) / "mlruns").as_uri())
    run_id = log_capital_run(carteira, sim, allocation=alloc, asrf=asrf,
                             run_name="capital_varejo_pf")
    run = mlflow.get_run(run_id)
    print("run_id:", run_id[:12])
    print("métricas:", {k: round(v, 1) for k, v in sorted(run.data.metrics.items())})

## 9. Fechamento — o método em 7 fases

O pacote cobre as fases de construção do guia. A recomendação operacional: **produto piloto de ponta a ponta,
v1 (ASRF) antes de sofisticar, frentes de parâmetros e de motor em paralelo**.

| Fase | Onde no pacote |
|---|---|
| 1. Definições (perda, horizonte, $q$, métrica) | parâmetros de `Portfolio.simulate` / `asrf_capital` |
| 2. Dados e segmentação | `Segment` / `Portfolio` (contrato homogêneo) |
| 3. Parâmetros (PD TTC, LGD/CCF downturn) | `parameters` (`pit_to_ttc`, `lgd_downturn_*`, `ccf_downturn`) |
| 4. Correlações e fatores | `correlation` (momentos, MLE, matriz entre fatores) |
| 5. Motor de cálculo (v1 e v2) | `asrf` · `monte_carlo` · `creditrisk_plus` · `migration` |
| 6. Validação e testes | `validation` (benchmark, sensibilidade, estresse, Pilar 1, convergência) |
| 7. Alocação, uso e governança | `allocation` (Euler, RAROC) · `tracking` (MLflow) |

**Próximos passos sugeridos:** calibrar os parâmetros dos seus segmentos reais, estimar $\rho$ das suas
séries de *default*, e paralelizar a simulação em PySpark/Databricks para a carteira completa.